------------------------------
#### Embedding as a text feature encoder for ML algorithms
------------------------------

In [1]:
import pandas as pd
import numpy as np
from ast import literal_eval

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, confusion_matrix, classification_report

In [2]:
datafile_path = r"D:\AI-DATASETS\02-MISC-large\GenAI-LLMs\amazon_food_reviews_with_embeddings_2k.csv"

In [3]:
df = pd.read_csv(datafile_path)

In [4]:
df.shape

(2000, 9)

In [5]:
df.sample(3)

,Unnamed: 0,ProductId,UserId,Score,Summary,Text,combined,n_tokens,ada_embedding
1340,196489,B002CJAT6W,ANP2X414OHQNS,3,damaged can in group,One of the cans arrived damaged with a collect...,Title: damaged can in group; Content: One of t...,74,"[0.03661475330591202, 0.015010871924459934, 0...."
474,209229,B003C6GFVW,A37C38YY5JLB9G,5,Great product,Product really works. I bought some for my fri...,Title: Great product; Content: Product really ...,40,"[0.020104827359318733, -0.024066101759672165, ..."
802,268027,B002YR97J2,A29IAD2FNB2RNE,5,I love TVP!,"Plain TVP, without flavoring or other ingredie...","Title: I love TVP!; Content: Plain TVP, withou...",165,"[-0.04919753223657608, -0.02480923756957054, -..."


**about literal_eval**

In [6]:
# Example 1: Safely evaluating a numeric literal
number_str = "42"
number = literal_eval(number_str)
print(number)  # Output: 42

# Example 2: Safely evaluating a list literal
list_str = "[1, 2, 3, 4]"
my_list = literal_eval(list_str)
print(my_list)  # Output: [1, 2, 3, 4]

# Example 3: Safely evaluating a dictionary literal
dict_str = "{'key': 'value', 'number': 123}"
my_dict = literal_eval(dict_str)
print(my_dict)  # Output: {'key': 'value', 'number': 123}

42
[1, 2, 3, 4]
{'key': 'value', 'number': 123}


In [7]:
df.dtypes

Unnamed: 0        int64
ProductId        object
UserId           object
Score             int64
Summary          object
Text             object
combined         object
n_tokens          int64
ada_embedding    object
dtype: object

In [8]:
df["embedding"] = df.ada_embedding.apply(literal_eval).apply(np.array)

In [9]:
df.dtypes

Unnamed: 0        int64
ProductId        object
UserId           object
Score             int64
Summary          object
Text             object
combined         object
n_tokens          int64
ada_embedding    object
embedding        object
dtype: object

In [10]:
df.sample(2)

,Unnamed: 0,ProductId,UserId,Score,Summary,Text,combined,n_tokens,ada_embedding,embedding
1730,384160,B000EVWQZW,A2PCNXBSKCABG5,4,Versatile Mix,This mix makes a good bread or can also be use...,Title: Versatile Mix; Content: This mix makes ...,56,"[0.007907040417194366, 0.013778990134596825, 0...","[0.007907040417194366, 0.013778990134596825, 0..."
1792,423200,B0055RU9RC,A1PLO6OW6O2Z1,5,Love it!,My dog has digestive issues and this product w...,Title: Love it!; Content: My dog has digestive...,74,"[-0.005880063399672508, -0.0002432850451441481...","[-0.005880063399672508, -0.0002432850451441481..."


In [9]:
list(df.embedding.values)[:3]

[array([ 0.01105704, -0.04999364, -0.07967817, ...,  0.04339708,
         0.00402546,  0.00280321]),
 array([ 0.00856422,  0.00123707, -0.06479561, ...,  0.01813236,
        -0.01135287, -0.01161934]),
 array([ 0.02059981,  0.0168439 , -0.03050874, ...,  0.01774122,
        -0.00544479,  0.02247136])]

In [12]:
list(df.embedding.values)[0].shape

(1536,)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(list(df.embedding.values), 
                                                        df.Score, 
                                                        test_size   = 0.2, 
                                                        random_state= 42)

**Logistic regression**

In [12]:
logr = LogisticRegression()
logr.fit(X_train, y_train)

LogisticRegression()

In [13]:
preds = logr.predict(X_test)

In [14]:
accuracy_score(y_test, preds), confusion_matrix(y_test, preds)

(0.7525,
 array([[ 37,   0,   0,   2,   4],
        [ 19,   1,   3,   5,   2],
        [  3,   0,   7,   8,   8],
        [  0,   0,   2,  13,  41],
        [  2,   0,   0,   0, 243]], dtype=int64))

In [17]:
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           1       0.60      0.86      0.70        43
           2       1.00      0.03      0.06        30
           3       0.64      0.27      0.38        26
           4       0.46      0.23      0.31        56
           5       0.82      0.99      0.90       245

    accuracy                           0.75       400
   macro avg       0.70      0.48      0.47       400
weighted avg       0.74      0.75      0.70       400



In [28]:
# mse = mean_squared_error(y_test, preds)
# mae = mean_absolute_error(y_test, preds)

# print(f"ada-002 embedding performance on 2k Amazon reviews: mse={mse:.2f}, mae={mae:.2f}")

We can see that the embeddings are able to predict the scores with an average error of 0.53 per score prediction. This is roughly equivalent to predicting half of reviews perfectly, and half off by one star.